In [2]:
import pandas as pd

In [17]:
gm12878_gwas = pd.read_csv(
    "../data/gwas/0048_Cells - Lymphoblastoid Cell_merged_loop.tsv",
    sep="\t",
)

gm12878_gwas = gm12878_gwas.dropna()
gm12878_gwas = gm12878_gwas[['bait_frag', 'other_frag', 'N_reads']]
gm12878_gwas['chrom_bait'] = gm12878_gwas['bait_frag'].str.split(',').str[0]
gm12878_gwas['start_bait'] = gm12878_gwas['bait_frag'].str.split(',').str[1].astype(int)
gm12878_gwas['end_bait'] = gm12878_gwas['bait_frag'].str.split(',').str[2].astype(int)
gm12878_gwas['chrom_other'] = gm12878_gwas['other_frag'].str.split(',').str[0]
gm12878_gwas['start_other'] = gm12878_gwas['other_frag'].str.split(',').str[1].astype(int)
gm12878_gwas['end_other'] = gm12878_gwas['other_frag'].str.split(',').str[2].astype(int)
gm12878_gwas = gm12878_gwas.drop(columns=['bait_frag', 'other_frag'])

gm12878_gwas['N_reads'] = gm12878_gwas['N_reads'].astype(int) - 2 # subtract 2 to offset minimum

gm12878_gwas = gm12878_gwas[['chrom_bait', 'start_bait', 'end_bait', 'chrom_other', 'start_other', 'end_other', 'N_reads']]

# output bedpe file
gm12878_gwas.to_csv("../data/gwas/gm12878_gwas_loops.bedpe", sep="\t", index=False, header=False)

In [18]:
gm12878_gwas = pd.read_csv(
    "../data/gwas/0048_Cells - Lymphoblastoid Cell_merged_loop.tsv",
    sep="\t",
)

gm12878_gwas = gm12878_gwas.dropna()
gm12878_bait_motifs = gm12878_gwas[['bait_frag', 'bait_gene_strand', 'score']].copy()
gm12878_bait_motifs['chrom'] = gm12878_bait_motifs['bait_frag'].str.split(',').str[0]
gm12878_bait_motifs['start'] = gm12878_bait_motifs['bait_frag'].str.split(',').str[1].astype(int)
gm12878_bait_motifs['end'] = gm12878_bait_motifs['bait_frag'].str.split(',').str[2].astype(int)
gm12878_bait_motifs['motif'] = gm12878_bait_motifs['bait_gene_strand'].map(lambda strand: dict((('+', 'p'), ('+,-', 'p'), ('-,+', 'n'), ('-', 'n')))[strand])
gm12878_bait_motifs = gm12878_bait_motifs[['chrom', 'start', 'end', 'motif', 'score']].drop_duplicates()


gm12878_other_motifs = gm12878_gwas[['other_frag', 'oe_gene_strand', 'score']].copy()
gm12878_other_motifs['chrom'] = gm12878_other_motifs['other_frag'].str.split(',').str[0]
gm12878_other_motifs['start'] = gm12878_other_motifs['other_frag'].str.split(',').str[1].astype(int)
gm12878_other_motifs['end'] = gm12878_other_motifs['other_frag'].str.split(',').str[2].astype(int)
gm12878_other_motifs['motif'] = gm12878_other_motifs['oe_gene_strand'].map(lambda strand: dict((('+', 'p'), ('+,-', 'p'), ('-,+', 'n'), ('-', 'n')))[strand])
gm12878_other_motifs = gm12878_other_motifs[['chrom', 'start', 'end', 'motif', 'score']].drop_duplicates()

gm12878_gwas_motifs = pd.concat([gm12878_bait_motifs, gm12878_other_motifs]).drop_duplicates()
# output bed file
gm12878_gwas_motifs.to_csv("../data/gwas/gm12878_gwas_motifs.bed", sep="\t", index=False, header=False)
gm12878_gwas_motifs

,chrom,start,end,motif,score
0,chr1,831895,848168,p,23.17
1,chr1,831895,848168,p,6.33
2,chr1,848169,850618,p,5.53
3,chr1,848169,850618,p,5.52
4,chr1,848169,850618,p,7.55
...,...,...,...,...,...
93879,chrX,154880070,154882443,n,5.42
93880,chrX,154456722,154462044,p,6.73
93882,chrX,154848133,154853541,n,5.62
93883,chrX,154357959,154364862,n,8.12


In [19]:
!pip install -r /cudammc/ccd-caller/requirements.txt
!cd /cudammc/ccd-caller && \
    /cudammc/ccd-caller/run.sh /home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe && \
    mv /cudammc/ccd-caller/output_hg38_PET3+_LEN100000-2000000 /home/jovyan/work/data/gwas/gm12878_gwas_ccd_calls

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 4.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 2.4 MB/s  0:00:23m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [numba]32m1/2 [numba]
Input file: /home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe
Using Python 3.12.7
=== Splitting input files into separate chromosomes and filtering data based on interaction length and PET-Count ===
    interaction length: 100000
    PET-Count limit:    3
    chr1 chr2 chr3 chr4 chr5 chr6 chr7 chr8 chr9 chr10 chr11 chr12 chr13 chr14 chr15 chr16 chr17 chr18 chr19 chr20 chr21 chr22 chrX 
=== Calculating expected coverage===
100%|█████████████████████████████████████| 3133/3133 [00:01<00:00, 2198.74it/s]
output_hg38_PET3+_LEN100000-2000000/expected_coverage_chr1_1000_hg38.bedgraph saved...
100%|█████████████████████████████████████| 1895/1895 [00:00<00:00, 1964.79it/s]
output_hg38_PET3+_LEN100000-2000000/expected_coverage_chr2_1000_hg38.bedgraph s

In [ ]:
!python /cudammc/ccd-caller/create_breakpoints.py \
    -i /home/jovyan/work/data/gwas/gm12878_gwas_ccd_calls/ccds_all_hg38.bed \
    -o /home/jovyan/work/data/gwas/gm12878_gwas_ccd_breakpoints.bed

In [21]:
!python /cudammc/ccd-caller/create_anchors_and_singletons.py \
    -i /home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe \
    -o /home/jovyan/work/data/gwas/gm12878_gwas_anchors \
    -c GM12878 \
    -f 3

/home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe /home/jovyan/work/data/gwas/gm12878_gwas_anchors GM12878 3
Executing cat /home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe | awk '($7>=3)' > /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_clusters_3+.bedpe
Executing cat /home/jovyan/work/data/gwas/gm12878_gwas_loops.bedpe | awk '($7<3)' > /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_singletons_lessthan3.bedpe
Executing cut -f1-3 /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_clusters_3+.bedpe > /home/jovyan/work/data/gwas/gm12878_gwas_anchors/a
Executing cut -f4-6 /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_clusters_3+.bedpe > /home/jovyan/work/data/gwas/gm12878_gwas_anchors/b
Executing cat /home/jovyan/work/data/gwas/gm12878_gwas_anchors/a /home/jovyan/work/data/gwas/gm12878_gwas_anchors/b | sort | awk '!visited[$0]++' | sortBed > /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_anchors_3+.bed
Executing rm /home/jovyan/work/dat

In [22]:
!pip install pybedtools
!python /cudammc/ccd-caller/create_oriented_anchors.py \
    -c /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_clusters_3+.bedpe \
    -a /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_anchors_3+.bed \
    -m /home/jovyan/work/data/gwas/gm12878_gwas_motifs.bed \
    -l GM12878 \
    -f 3 \
    -o /home/jovyan/work/data/gwas/gm12878_gwas_oriented_anchors

/home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_clusters_3+.bedpe /home/jovyan/work/data/gwas/gm12878_gwas_anchors/GM12878_anchors_3+.bed /home/jovyan/work/data/gwas/gm12878_gwas_motifs.bed GM12878 3 /home/jovyan/work/data/gwas/gm12878_gwas_oriented_anchors
stage1
stage2
stage3
stage4
stage5
stage6
stage7
/cudammc/ccd-caller/create_oriented_anchors.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged_anchors_ctcf_motif_filterCols['anchor_name'] = anchor_name
stage8
